# Day 4: Weaviate
## Semantic Research Paper Discovery

Weaviate is an open-source vector database with a managed cloud offering. Its defining feature is hybrid search - the ability to combine vector similarity, BM25 keyword matching and metadata filtering in a single query. This makes it particularly well suited to knowledge-heavy retrieval tasks where meaning and keywords both matter.

**When would you reach for this?**
- You need hybrid search combining semantic similarity with keyword matching
- Your data has rich structured metadata alongside free-text content
- You want a schema-aware vector database with strong typing

**The use case:** A research paper discovery system where users can find relevant papers by describing their topic of interest in natural language, optionally filtered by research field, publication year or citation count. Hybrid search ensures that both the meaning of the query and specific technical terms are captured in the results.

## 1. Setup

### Prerequisites

- A Weaviate Cloud account
- Ollama running locally with the `all-minilm` model pulled
- Python 3.12 with a virtual environment

### Create a Free Weaviate Cloud Cluster

If you do not already have a cluster:

1. Go to the [Weaviate Console](https://console.weaviate.cloud) and create an account
2. Click **Create new cluster > Free**
3. Give the cluster a name (e.g. `papers`)
4. Accept all the other default options and and click **Create cluster**
5. Click the **How to connect** button and note down the `WEAVIATE_URL`
6. From the cluster page select **API Keys**, create a new **Admin** key and note it down

### Set Environment Variables

Set your Weaviate credentials as environment variables before running the notebook:

```shell
export WEAVIATE_URL="your-cluster-name.c0.region.cloud-provider.weaviate.cloud"
export WEAVIATE_API_KEY="your-api-key"
```

### Install Python Dependencies

In [1]:
%pip install ollama==0.6.2 \
             pandas==3.0.3 \
             protobuf==5.29.4 \
             tqdm==4.67.1 \
             weaviate-client==4.22.0 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama
import os
import pandas as pd
import random
import weaviate
import weaviate.classes as wvc

from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter, HybridFusion
from tqdm.notebook import tqdm

### Configuration

In [3]:
WEAVIATE_URL     = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]
COLLECTION_NAME  = "ResearchPaper"
LLM_EMBEDDING    = "all-minilm"
NUM_PAPERS       = 200
RANDOM_SEED      = 42

> **Note:** `NUM_PAPERS` controls the size of the generated dataset. 200 is the recommended default for this tutorial. Embedding generation runs locally via Ollama and is single-threaded. At 200 papers the notebook runs comfortably. Larger values will work but will take proportionally longer.

### Verify Ollama is Running

In [4]:
ollama_ready = False

try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    assert any(LLM_EMBEDDING in m for m in model_names)
    print(f"Model '{LLM_EMBEDDING}' is ready.")
    ollama_ready = True
except ConnectionError:
    print("ERROR: Ollama is not running. Start it with: ollama serve")
except AssertionError:
    print(f"ERROR: Model not found. Run: ollama pull {LLM_EMBEDDING}")

Model 'all-minilm' is ready.


In [5]:
assert ollama_ready, "Please fix the Ollama issue above before continuing."

### Determine Embedding Dimensions

In [6]:
def get_embedding(text: str) -> list:
    response = ollama.embeddings(model = LLM_EMBEDDING, prompt = text)
    return response["embedding"]

test_embedding = get_embedding("transformer architecture for natural language processing")
EMBEDDING_DIMS = len(test_embedding)
print(f"Embedding dimensions: {EMBEDDING_DIMS}")

Embedding dimensions: 384


### Connect to Weaviate Cloud

In [7]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url = WEAVIATE_URL,
    auth_credentials = Auth.api_key(WEAVIATE_API_KEY)
)

print(f"Connected to Weaviate: {client.is_ready()}")

Connected to Weaviate: True


## 2. The Dataset

We generate research papers programmatically from pools of fields, venues, authors and abstract templates. Each abstract is assembled from domain-appropriate components, giving enough variation for meaningful hybrid search.

The `abstract` field is what we embed and search over. The structured fields - `field`, `year`, `venue` and `citation_count` - serve as filters.

In [8]:
random.seed(RANDOM_SEED)

FIELDS = [
    "Machine Learning", "Computer Vision", "Natural Language Processing",
    "Reinforcement Learning", "Graph Neural Networks", "Robotics",
    "Bioinformatics", "Quantum Computing", "Cybersecurity", "Data Engineering",
]

VENUES = [
    "International Conference on Machine Learning Systems",
    "Journal of Advanced Artificial Intelligence",
    "Proceedings of the Data Science Symposium",
    "International Workshop on Neural Computing",
    "Journal of Computational Intelligence Research",
    "Transactions on Machine Learning and Data Mining",
    "Conference on AI Systems and Applications",
    "International Journal of Deep Learning",
    "Symposium on Knowledge Discovery and Data Mining",
    "Workshop on Advances in Neural Information Processing",
    "Journal of Applied Machine Learning",
    "Conference on Intelligent Data Analysis",
]

FIRST_NAMES = [
    "James", "Wei", "Sarah", "Ahmed", "Priya", "Lena", "Carlos",
    "Yuki", "Emma", "Ali", "Chen", "Sofia", "David", "Fatima", "Raj",
]

LAST_NAMES = [
    "Zhang", "Kumar", "Johnson", "Mueller", "Patel", "Kim", "Garcia",
    "Tanaka", "Smith", "Hassan", "Li", "Rossi", "Brown", "Nguyen", "Singh",
]

METHODS = {
    "Machine Learning":             ["gradient boosting", "neural architecture search", "meta-learning",
                                     "self-supervised learning", "federated learning", "transfer learning"],
    "Computer Vision":              ["convolutional neural networks", "vision transformers", "object detection",
                                     "image segmentation", "generative adversarial networks", "depth estimation"],
    "Natural Language Processing":  ["transformer models", "large language models", "named entity recognition",
                                     "question answering", "machine translation", "sentiment analysis"],
    "Reinforcement Learning":       ["policy gradient methods", "model-based RL", "multi-agent systems",
                                     "reward shaping", "hierarchical RL", "offline RL"],
    "Graph Neural Networks":        ["message passing networks", "graph attention networks", "graph transformers",
                                     "knowledge graph embeddings", "link prediction", "node classification"],
    "Robotics":                     ["motion planning", "sim-to-real transfer", "imitation learning",
                                     "manipulation tasks", "autonomous navigation", "tactile sensing"],
    "Bioinformatics":               ["protein structure prediction", "genomic sequence analysis", "drug discovery",
                                     "single-cell RNA sequencing", "molecular docking", "gene expression"],
    "Quantum Computing":            ["quantum circuit optimization", "variational quantum algorithms",
                                     "quantum error correction", "quantum machine learning", "qubit encoding"],
    "Cybersecurity":                ["adversarial machine learning", "intrusion detection", "malware classification",
                                     "federated privacy", "anomaly detection", "cryptographic protocols"],
    "Data Engineering":             ["stream processing", "data lake architectures", "query optimization",
                                     "vector databases", "ETL pipelines", "data versioning"],
}

DATASETS = {
    "Machine Learning":             ["ImageNet", "CIFAR-10", "UCI repository", "OpenML benchmarks"],
    "Computer Vision":              ["COCO", "Pascal VOC", "ADE20K", "CelebA", "Waymo Open Dataset"],
    "Natural Language Processing":  ["GLUE", "SuperGLUE", "SQuAD", "CommonCrawl", "BooksCorpus"],
    "Reinforcement Learning":       ["Atari", "MuJoCo", "OpenAI Gym", "D4RL", "StarCraft II"],
    "Graph Neural Networks":        ["Cora", "Citeseer", "OGB benchmarks", "Freebase", "ZINC"],
    "Robotics":                     ["RoboNet", "D4RL", "Meta-World", "RLBench", "Isaac Gym"],
    "Bioinformatics":               ["UniProt", "PDB", "TCGA", "GTEx", "ChEMBL"],
    "Quantum Computing":            ["IBM Quantum", "Cirq benchmarks", "QuTiP simulations"],
    "Cybersecurity":                ["NSL-KDD", "CICIDS", "VirusTotal", "EMBER", "PhishTank"],
    "Data Engineering":             ["TPC-H", "TPC-DS", "Stack Overflow", "NYC Taxi", "GitHub Archive"],
}

CONTRIBUTIONS = [
    "state-of-the-art results on",
    "significant improvements over existing baselines on",
    "competitive performance on",
    "novel insights into",
    "a new benchmark for",
]

ABSTRACT_TEMPLATES = [
    "We present a new approach to {field} using {method}. Our method addresses key limitations "
    "of prior work by introducing a novel architecture trained on {dataset}. "
    "Experiments demonstrate {contribution} standard benchmarks, with ablation studies "
    "confirming the importance of each component.",

    "This paper proposes {method} for {field} tasks. We identify fundamental challenges "
    "in existing approaches and design a scalable solution evaluated on {dataset}. "
    "Our approach achieves {contribution} multiple evaluation metrics.",

    "We investigate the application of {method} to {field}. Through extensive experiments "
    "on {dataset}, we demonstrate {contribution} prior methods while reducing "
    "computational cost. We release our code and models publicly.",

    "Recent advances in {field} have been driven by {method}. We build on these foundations "
    "to propose an improved framework evaluated on {dataset}. "
    "Results show {contribution} across diverse settings and ablation studies validate our design choices.",

    "The problem of {field} remains challenging due to data scarcity and model complexity. "
    "We address this with {method}, validated on {dataset}. "
    "Our work shows {contribution} and opens new directions for future research.",
]

TITLE_TEMPLATES = [
    "{Method} for {Field}: A Scalable Approach",
    "Improving {Field} with {Method}",
    "Towards Better {Field} via {Method}",
    "A Unified Framework for {Field} using {Method}",
    "{Method} in {Field}: Challenges and Opportunities",
    "Rethinking {Field} with {Method}",
    "Efficient {Method} for {Field}",
]

def generate_authors(n: int = None) -> list:
    n = n or random.randint(2, 5)
    return [
        f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
        for _ in range(n)
    ]

def generate_paper() -> dict:
    field   = random.choice(FIELDS)
    method  = random.choice(METHODS[field])
    dataset = random.choice(DATASETS[field])

    title = random.choice(TITLE_TEMPLATES).format(
        Method = method.title(),
        Field  = field,
    )

    abstract = random.choice(ABSTRACT_TEMPLATES).format(
        field        = field,
        method       = method,
        dataset      = dataset,
        contribution = random.choice(CONTRIBUTIONS),
    )

    return {
        "title":          title,
        "authors":        generate_authors(),
        "year":           random.randint(2015, 2024),
        "field":          field,
        "venue":          random.choice(VENUES),
        "citation_count": random.randint(0, 500),
        "abstract":       abstract,
    }

papers = [generate_paper() for _ in range(NUM_PAPERS)]
df = pd.DataFrame(papers)

print(f"Generated {len(df)} research papers")
df.head()

Generated 200 research papers


,title,authors,year,field,venue,citation_count,abstract
0,Improving Computer Vision with Convolutional N...,"[Chen Rossi, Raj Smith]",2016,Computer Vision,Workshop on Advances in Neural Information Pro...,216,This paper proposes convolutional neural netwo...
1,Improving Machine Learning with Gradient Boosting,"[Emma Mueller, Sofia Li]",2023,Machine Learning,Conference on AI Systems and Applications,112,This paper proposes gradient boosting for Mach...
2,Efficient Qubit Encoding for Quantum Computing,"[Lena Patel, Sarah Mueller, David Kim, Wei Kum...",2020,Quantum Computing,Transactions on Machine Learning and Data Mining,309,We present a new approach to Quantum Computing...
3,Message Passing Networks in Graph Neural Netwo...,"[Emma Patel, Fatima Li]",2024,Graph Neural Networks,Transactions on Machine Learning and Data Mining,295,We present a new approach to Graph Neural Netw...
4,Offline Rl for Reinforcement Learning: A Scala...,"[Fatima Mueller, Fatima Kumar]",2021,Reinforcement Learning,Journal of Computational Intelligence Research,232,This paper proposes offline RL for Reinforceme...


## 3. Create the Weaviate Collection

Weaviate organizes data in **collections**. Each collection has a defined schema with typed properties. We specify `NONE` as the vectorizer since we are providing our own embeddings from Ollama rather than using a built-in Weaviate vectorizer.

If the collection already exists from a previous run, we delete it and recreate it for a clean start.

In [9]:
if client.collections.exists(COLLECTION_NAME):
    print(f"Deleting existing collection '{COLLECTION_NAME}'...")
    client.collections.delete(COLLECTION_NAME)
    print("Deleted.")

collection = client.collections.create(
    name = COLLECTION_NAME,
    vector_config = Configure.Vectors.self_provided(
        vector_index_config = Configure.VectorIndex.hfresh(
            distance_metric = wvc.config.VectorDistances.COSINE
        )
    ),
    properties = [
        Property(name = "title",          data_type = DataType.TEXT),
        Property(name = "authors",        data_type = DataType.TEXT_ARRAY),
        Property(name = "year",           data_type = DataType.INT),
        Property(name = "field",          data_type = DataType.TEXT),
        Property(name = "venue",          data_type = DataType.TEXT),
        Property(name = "citation_count", data_type = DataType.INT),
        Property(name = "abstract",       data_type = DataType.TEXT),
    ]
)

print(f"Collection '{COLLECTION_NAME}' created.")

Deleting existing collection 'ResearchPaper'...
Deleted.
Collection 'ResearchPaper' created.


## 4. Generate Embeddings and Load Data

We embed each abstract and insert the paper along with its embedding into Weaviate. We use Weaviate's batch insert for efficiency.

In [10]:
collection = client.collections.get(COLLECTION_NAME)

with collection.batch.dynamic() as batch:
    for paper in tqdm(papers, desc = "Inserting papers"):
        embedding = get_embedding(paper["abstract"])
        batch.add_object(
            properties = {
                "title":          paper["title"],
                "authors":        paper["authors"],
                "year":           paper["year"],
                "field":          paper["field"],
                "venue":          paper["venue"],
                "citation_count": paper["citation_count"],
                "abstract":       paper["abstract"],
            },
            vector = embedding
        )

print(f"\nInserted {collection.aggregate.over_all().total_count} papers.")

Inserting papers:   0%|          | 0/200 [00:00<?, ?it/s]


Inserted 200 papers.


## 5. Semantic Search

We start with pure vector search to establish a baseline before introducing hybrid search.

In [11]:
def search_papers(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)

    results = collection.query.near_vector(
        near_vector      = query_embedding,
        limit            = top_k,
        return_metadata  = MetadataQuery(distance = True),
    )

    print(f"\nQuery: '{query}'\n")
    for obj in results.objects:
        p = obj.properties
        print(f"  {p['title']}")
        print(f"  {p['field']} | {p['venue']} | {p['year']} | Citations: {p['citation_count']}")
        print(f"  Authors: {', '.join(p['authors'])}")
        print(f"  Distance: {obj.metadata.distance:.3f}")
        print()

In [12]:
search_papers("transformer models for natural language understanding")


Query: 'transformer models for natural language understanding'

  Efficient Transformer Models for Natural Language Processing
  Natural Language Processing | Transactions on Machine Learning and Data Mining | 2020 | Citations: 196
  Authors: Sarah Kumar, Ahmed Brown
  Distance: 0.241

  Efficient Transformer Models for Natural Language Processing
  Natural Language Processing | Symposium on Knowledge Discovery and Data Mining | 2022 | Citations: 18
  Authors: Chen Kumar, Fatima Kim, Fatima Li, Fatima Smith, Carlos Kim
  Distance: 0.272

  Towards Better Natural Language Processing via Transformer Models
  Natural Language Processing | Proceedings of the Data Science Symposium | 2018 | Citations: 255
  Authors: Ahmed Nguyen, Raj Johnson, Fatima Kim, Raj Hassan, Sofia Kim
  Distance: 0.300

  Improving Natural Language Processing with Transformer Models
  Natural Language Processing | Journal of Applied Machine Learning | 2016 | Citations: 293
  Authors: Chen Mueller, David Patel, Lena

In [13]:
search_papers("reinforcement learning for robotic manipulation")


Query: 'reinforcement learning for robotic manipulation'

  Imitation Learning for Robotics: A Scalable Approach
  Robotics | Symposium on Knowledge Discovery and Data Mining | 2023 | Citations: 208
  Authors: Lena Brown, Priya Kumar, Sarah Kumar, Sarah Garcia, Yuki Smith
  Distance: 0.376

  Model-Based Rl for Reinforcement Learning: A Scalable Approach
  Reinforcement Learning | International Journal of Deep Learning | 2021 | Citations: 204
  Authors: Ali Nguyen, David Kumar, Yuki Garcia, Raj Li, Ali Mueller
  Distance: 0.422

  Efficient Motion Planning for Robotics
  Robotics | Proceedings of the Data Science Symposium | 2020 | Citations: 39
  Authors: Ali Garcia, Emma Kumar, Carlos Nguyen, Priya Rossi, Lena Mueller
  Distance: 0.423

  Towards Better Reinforcement Learning via Policy Gradient Methods
  Reinforcement Learning | Symposium on Knowledge Discovery and Data Mining | 2024 | Citations: 305
  Authors: Ahmed Mueller, Sofia Tanaka, Priya Rossi
  Distance: 0.424

  Manipulat

In [14]:
search_papers("graph neural networks for molecular property prediction")


Query: 'graph neural networks for molecular property prediction'

  Rethinking Graph Neural Networks with Link Prediction
  Graph Neural Networks | Workshop on Advances in Neural Information Processing | 2017 | Citations: 291
  Authors: Sofia Patel, Emma Johnson, Ahmed Garcia, Chen Garcia, Chen Rossi
  Distance: 0.364

  Improving Graph Neural Networks with Node Classification
  Graph Neural Networks | International Workshop on Neural Computing | 2019 | Citations: 196
  Authors: Ali Patel, Chen Garcia, Sofia Patel, Yuki Nguyen
  Distance: 0.379

  Efficient Link Prediction for Graph Neural Networks
  Graph Neural Networks | International Workshop on Neural Computing | 2024 | Citations: 331
  Authors: Yuki Zhang, Ali Singh, Emma Kim
  Distance: 0.387

  Link Prediction in Graph Neural Networks: Challenges and Opportunities
  Graph Neural Networks | Workshop on Advances in Neural Information Processing | 2021 | Citations: 157
  Authors: Sofia Hassan, Priya Tanaka, Fatima Zhang, Lena Kim

## 6. Hybrid Search

Hybrid search combines vector similarity with BM25 keyword matching in a single query. The `alpha` parameter controls the blend:

- `alpha = 0.0` - pure BM25 keyword search
- `alpha = 0.5` - balanced blend of BM25 and vector search
- `alpha = 1.0` - pure vector search

For research papers, a balanced blend works well because users often include specific technical terms ("BERT", "GAN", "HNSW") that benefit from keyword matching alongside the semantic meaning of the query.

In [15]:
def hybrid_search_papers(query: str, alpha: float = 0.5, top_k: int = 5):
    query_embedding = get_embedding(query)

    results = collection.query.hybrid(
        query           = query,
        vector          = query_embedding,
        alpha           = alpha,
        limit           = top_k,
        fusion_type     = HybridFusion.RELATIVE_SCORE,
        return_metadata = MetadataQuery(score = True),
    )

    print(f"\nHybrid search: '{query}' (alpha = {alpha})\n")
    for obj in results.objects:
        p = obj.properties
        print(f"  {p['title']}")
        print(f"  {p['field']} | {p['venue']} | {p['year']} | Citations: {p['citation_count']}")
        print(f"  Score: {obj.metadata.score:.3f}")
        print()

In [16]:
hybrid_search_papers("transformer models for natural language understanding", alpha = 0.5)


Hybrid search: 'transformer models for natural language understanding' (alpha = 0.5)

  Efficient Transformer Models for Natural Language Processing
  Natural Language Processing | Transactions on Machine Learning and Data Mining | 2020 | Citations: 196
  Score: 0.977

  Efficient Transformer Models for Natural Language Processing
  Natural Language Processing | Symposium on Knowledge Discovery and Data Mining | 2022 | Citations: 18
  Score: 0.975

  Improving Natural Language Processing with Transformer Models
  Natural Language Processing | Journal of Applied Machine Learning | 2016 | Citations: 293
  Score: 0.914

  Towards Better Natural Language Processing via Transformer Models
  Natural Language Processing | Proceedings of the Data Science Symposium | 2018 | Citations: 255
  Score: 0.909

  Transformer Models in Natural Language Processing: Challenges and Opportunities
  Natural Language Processing | International Journal of Deep Learning | 2024 | Citations: 337
  Score: 0.874


### Comparing alpha values

The same query with different alpha values shows how the blend shifts results. A lower alpha favors papers that contain the exact query terms; a higher alpha favors papers that are semantically similar regardless of specific wording.

In [17]:
query = "large language models for question answering"

for alpha in [0.0, 0.5, 1.0]:
    hybrid_search_papers(query, alpha = alpha, top_k = 3)


Hybrid search: 'large language models for question answering' (alpha = 0.0)

  Towards Better Natural Language Processing via Question Answering
  Natural Language Processing | Journal of Advanced Artificial Intelligence | 2024 | Citations: 269
  Score: 1.000

  Efficient Question Answering for Natural Language Processing
  Natural Language Processing | Transactions on Machine Learning and Data Mining | 2018 | Citations: 485
  Score: 0.999

  Towards Better Natural Language Processing via Large Language Models
  Natural Language Processing | Workshop on Advances in Neural Information Processing | 2018 | Citations: 165
  Score: 0.825


Hybrid search: 'large language models for question answering' (alpha = 0.5)

  Efficient Question Answering for Natural Language Processing
  Natural Language Processing | Transactions on Machine Learning and Data Mining | 2018 | Citations: 485
  Score: 0.999

  Towards Better Natural Language Processing via Question Answering
  Natural Language Processi

## 7. Filtered Hybrid Search

Weaviate allows filters to be combined with hybrid search. Filters apply to the collection properties and are expressed using Weaviate's `Filter` class. Unlike Pinecone, filter fields do not need to be declared separately - any property defined in the collection schema can be used as a filter.

In [18]:
def hybrid_search_filtered(
    query:            str,
    alpha:            float = 0.5,
    field:            str   = None,
    min_year:         int   = None,
    min_citations:    int   = None,
    top_k:            int   = 5
):
    query_embedding = get_embedding(query)

    # Build filter chain
    filters = None
    if field:
        f = Filter.by_property("field").equal(field)
        filters = f if filters is None else filters & f
    if min_year:
        f = Filter.by_property("year").greater_or_equal(min_year)
        filters = f if filters is None else filters & f
    if min_citations:
        f = Filter.by_property("citation_count").greater_or_equal(min_citations)
        filters = f if filters is None else filters & f

    results = collection.query.hybrid(
        query        = query,
        vector       = query_embedding,
        alpha        = alpha,
        limit        = top_k,
        fusion_type  = HybridFusion.RELATIVE_SCORE,
        filters      = filters,
        return_metadata = MetadataQuery(score = True),
    )

    label = f"query = '{query}', alpha = {alpha}"
    if field:         label += f", field = '{field}'"
    if min_year:      label += f", min_year = {min_year}"
    if min_citations: label += f", min_citations = {min_citations}"
    print(f"\n{label}\n")

    for obj in results.objects:
        p = obj.properties
        print(f"  {p['title']}")
        print(f"  {p['field']} | {p['venue']} | {p['year']} | Citations: {p['citation_count']}")
        print(f"  Score: {obj.metadata.score:.3f}")
        print()

In [19]:
# NLP papers from 2020 onwards with at least 100 citations
hybrid_search_filtered(
    "attention mechanisms and transformer architectures",
    field         = "Natural Language Processing",
    min_year      = 2020,
    min_citations = 100
)


query = 'attention mechanisms and transformer architectures', alpha = 0.5, field = 'Natural Language Processing', min_year = 2020, min_citations = 100

  Efficient Transformer Models for Natural Language Processing
  Natural Language Processing | Transactions on Machine Learning and Data Mining | 2020 | Citations: 196
  Score: 0.918

  Transformer Models in Natural Language Processing: Challenges and Opportunities
  Natural Language Processing | International Journal of Deep Learning | 2024 | Citations: 337
  Score: 0.500

  Efficient Machine Translation for Natural Language Processing
  Natural Language Processing | Transactions on Machine Learning and Data Mining | 2024 | Citations: 442
  Score: 0.298

  Towards Better Natural Language Processing via Question Answering
  Natural Language Processing | Journal of Advanced Artificial Intelligence | 2024 | Citations: 269
  Score: 0.228

  Efficient Sentiment Analysis for Natural Language Processing
  Natural Language Processing | Intern

In [20]:
# Recent computer vision papers
hybrid_search_filtered(
    "object detection in real time",
    field    = "Computer Vision",
    min_year = 2022
)


query = 'object detection in real time', alpha = 0.5, field = 'Computer Vision', min_year = 2022

  Rethinking Computer Vision with Object Detection
  Computer Vision | International Journal of Deep Learning | 2024 | Citations: 269
  Score: 1.000

  Towards Better Computer Vision via Vision Transformers
  Computer Vision | Conference on Intelligent Data Analysis | 2022 | Citations: 304
  Score: 0.440

  Towards Better Computer Vision via Convolutional Neural Networks
  Computer Vision | International Workshop on Neural Computing | 2023 | Citations: 292
  Score: 0.337

  Efficient Depth Estimation for Computer Vision
  Computer Vision | International Conference on Machine Learning Systems | 2023 | Citations: 306
  Score: 0.156

  Rethinking Computer Vision with Depth Estimation
  Computer Vision | Journal of Advanced Artificial Intelligence | 2022 | Citations: 49
  Score: 0.000



In [21]:
# Highly cited papers across all fields
hybrid_search_filtered(
    "self-supervised learning for representation learning",
    min_citations = 200
)


query = 'self-supervised learning for representation learning', alpha = 0.5, min_citations = 200

  Self-Supervised Learning in Machine Learning: Challenges and Opportunities
  Machine Learning | Workshop on Advances in Neural Information Processing | 2019 | Citations: 348
  Score: 0.918

  A Unified Framework for Machine Learning using Self-Supervised Learning
  Machine Learning | Workshop on Advances in Neural Information Processing | 2017 | Citations: 235
  Score: 0.915

  Improving Machine Learning with Self-Supervised Learning
  Machine Learning | Transactions on Machine Learning and Data Mining | 2024 | Citations: 295
  Score: 0.881

  Efficient Federated Learning for Machine Learning
  Machine Learning | Journal of Advanced Artificial Intelligence | 2018 | Citations: 362
  Score: 0.453

  Towards Better Machine Learning via Meta-Learning
  Machine Learning | Workshop on Advances in Neural Information Processing | 2016 | Citations: 379
  Score: 0.412



## Cleanup

In [22]:
client.close()

print("Connection closed.")

Connection closed.
